# Multi-Modal Violence Detection: Audio Module
## Binary Classification (Violence vs. Non-Violence)
This notebook contains the complete audio pipeline. Run these cells in order.

In [ ]:
# 1. Environment Setup
!pip install torchaudio librosa pandas numpy
import torch
import torchaudio
import librosa
import pandas as pd
import torch.nn as nn
import torch.optim as optim
print('Libraries Imported Successfully! PyTorch version:', torch.__version__)

### 1. The Audio Slicer & Formatter
Contains the logic to format audio to exact 5-second chunks.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class ViolenceAudioDataset(Dataset):
    # Setup for loading datasets later when you mount Google Drive
    pass # Full code will be pasted here when we integrate Drive

### 2. The Sound-to-Image Converter (Log-Mel Spectrogram)

In [ ]:
class LogMelSpectrogramFeatureExtractor:
    def __init__(self, sample_rate=32000, n_fft=1024, hop_length=320, n_mels=64):
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)

    def __call__(self, waveform):
        return self.amplitude_to_db(self.mel_spectrogram(waveform))

### 3. The Alarm Bell (Detection Model)

In [ ]:
class AudioViolenceClassifier(nn.Module):
    def __init__(self, num_classes=1):
        super(AudioViolenceClassifier, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(nn.Linear(128, num_classes), nn.Sigmoid())

    def forward(self, x):
        if x.dim() == 3: x = x.unsqueeze(1)
        x = self.conv(x).view(x.size(0), -1)
        return self.fc(x)

### 4. Test The Pipeline (Dummy Data)

In [ ]:
# Let's test it completely with dummy data right now
model = AudioViolenceClassifier()
extractor = LogMelSpectrogramFeatureExtractor()

dummy_audio = torch.randn(4, 1, 32000 * 5) # 4 random 5-second audio clips
dummy_labels = torch.tensor([[1.0], [0.0], [1.0], [0.0]]) # 1=Violence, 0=Normal

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

for epoch in range(3):
    optimizer.zero_grad()
    spectrograms = extractor(dummy_audio.squeeze(1))
    predictions = model(spectrograms)
    loss = criterion(predictions, dummy_labels)
    loss.backward()
    optimizer.step()
    
    print(f'Epoch {epoch+1} - Error Loss: {loss.item():.4f}')
